# Feature Engineering — T1/T2 head-to-head format

Reads `matchups.csv` which has **one row per matchup** with `T1_*` and `T2_*` side-by-side columns.

Adds:
- `Diff_*` features = `T1_stat − T2_stat` for every numeric T1/T2 pair
- `TeamID` and `OppID` identifiers
- `Elo_Win_Prob` from `Diff_Pregame_Elo` (if available)

Output: **`matchups_v2.csv`**

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

ELO_HOME_ADV = 100.0

# Find matchups.csv
cwd = Path(os.getcwd())
print(f'Working directory: {cwd}')

csv_path = None
for candidate in ['matchups.csv', '../matchups.csv', 'data/matchups.csv']:
    if Path(candidate).exists():
        csv_path = candidate
        break

if csv_path is None:
    raise FileNotFoundError('matchups.csv not found. Run build_dataset.ipynb first.')

mm = pd.read_csv(csv_path, low_memory=False)
print(f'Loaded: {csv_path}  shape={mm.shape}')
print(f'\nAll columns ({len(mm.columns)}):')
print(list(mm.columns))

## Detect Format & Normalize Columns

In [ ]:
def find_col(df, *candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Detect T1/T2 format vs team-perspective format
t1_cols = [c for c in mm.columns if c.startswith('T1_')]
t2_cols = [c for c in mm.columns if c.startswith('T2_')]
has_t1t2 = len(t1_cols) > 0 and len(t2_cols) > 0

print(f'Format detected: {"T1/T2 head-to-head" if has_t1t2 else "team-perspective"}' )
print(f'  T1_* columns: {len(t1_cols)}')
print(f'  T2_* columns: {len(t2_cols)}')

# Find key columns under various naming conventions
season_col  = find_col(mm, 'Season', 'SEASON', 'season', 'Year')
gameid_col  = find_col(mm, 'GameID', 'GAME_ID', 'game_id', 'ID')
date_col    = find_col(mm, 'DayDate', 'GAME_DATE', 'Date', 'date')
target_col  = find_col(mm, 'Target_Win', 'WL', 'W', 'WIN', 'Home_Win')
is_home_col = find_col(mm, 'Is_Home', 'HOME', 'is_home', 'IS_HOME')
elo_col     = find_col(mm, 'Diff_Pregame_Elo', 'Diff_ELO', 'ELO_DIFF', 'elo_diff')

# T1/T2 team ID columns
t1_id_col = find_col(mm, 'T1_TeamID', 'T1_TEAM_ID', 'T1_ID', 'T1_team_id')
t2_id_col = find_col(mm, 'T2_TeamID', 'T2_TEAM_ID', 'T2_ID', 'T2_team_id')

print(f'\nKey columns found:')
print(f'  Season:         {season_col}')
print(f'  GameID:         {gameid_col}')
print(f'  Date:           {date_col}')
print(f'  Target_Win:     {target_col}')
print(f'  Is_Home:        {is_home_col}')
print(f'  Diff_Pregame_Elo: {elo_col}')
print(f'  T1 team ID:     {t1_id_col}')
print(f'  T2 team ID:     {t2_id_col}')

# Standardize key column names
rename = {}
if season_col  and season_col  != 'Season':    rename[season_col]  = 'Season'
if gameid_col  and gameid_col  != 'GameID':    rename[gameid_col]  = 'GameID'
if date_col    and date_col    != 'DayDate':   rename[date_col]    = 'DayDate'
if target_col  and target_col  != 'Target_Win':rename[target_col]  = 'Target_Win'
if is_home_col and is_home_col != 'Is_Home':   rename[is_home_col] = 'Is_Home'
if elo_col     and elo_col     != 'Diff_Pregame_Elo': rename[elo_col] = 'Diff_Pregame_Elo'
if t1_id_col   and t1_id_col   != 'T1_TeamID': rename[t1_id_col]  = 'T1_TeamID'
if t2_id_col   and t2_id_col   != 'T2_TeamID': rename[t2_id_col]  = 'T2_TeamID'

if rename:
    mm = mm.rename(columns=rename)
    print(f'\nRenamed: {rename}')

# Parse date
if 'DayDate' in mm.columns:
    mm['DayDate'] = pd.to_datetime(mm['DayDate'], errors='coerce')

# Convert WL strings to 0/1
if 'Target_Win' in mm.columns:
    col = mm['Target_Win']
    if col.dtype == object:
        mm['Target_Win'] = col.map({'W':1,'L':0,'w':1,'l':0}).fillna(col).astype(float)

# Ensure Is_Home exists
if 'Is_Home' not in mm.columns:
    mm['Is_Home'] = 1  # assume T1 is home (common convention)
    print('Is_Home not found — assuming T1 is home team')

# Ensure Diff_Pregame_Elo exists
if 'Diff_Pregame_Elo' not in mm.columns:
    mm['Diff_Pregame_Elo'] = 0.0
    print('Diff_Pregame_Elo not found — set to 0 (ELO baseline will be 50/50)')

print(f'\nDataframe ready: {mm.shape}')

## Build TeamID / OppID

In T1/T2 format, `T1_*` = this team's stats, `T2_*` = opponent's stats.
We assign `TeamID = T1` identifier and `OppID = T2` identifier.

In [ ]:
# Create TeamID and OppID
if 'TeamID' not in mm.columns:
    if 'T1_TeamID' in mm.columns:
        mm['TeamID'] = mm['T1_TeamID']
        print('TeamID ← T1_TeamID')
    elif 'GameID' in mm.columns:
        # Use sequential row-based ID as a proxy
        mm['TeamID'] = mm.index.astype(str) + '_T1'
        print('TeamID synthesized from row index (no T1_TeamID found)')
    else:
        mm['TeamID'] = range(len(mm))
        print('TeamID: sequential integers')

if 'OppID' not in mm.columns:
    if 'T2_TeamID' in mm.columns:
        mm['OppID'] = mm['T2_TeamID']
        print('OppID  ← T2_TeamID')
    elif 'TeamID' in mm.columns:
        mm['OppID'] = mm['TeamID'].astype(str) + '_opp'
        print('OppID synthesized')
    else:
        mm['OppID'] = range(len(mm))

print(f'\nTeamID sample: {mm["TeamID"].unique()[:5]}')
print(f'OppID  sample: {mm["OppID"].unique()[:5]}')

## Compute Diff_ Features from T1/T2 Pairs

For every `T1_X` column that has a matching `T2_X`, compute `Diff_X = T1_X − T2_X`.

In [ ]:
# Find all T1/T2 numeric pairs and compute diffs
t1_cols = [c for c in mm.columns if c.startswith('T1_') and pd.api.types.is_numeric_dtype(mm[c])]
new_diff_cols = []

for t1_col in t1_cols:
    stat_name = t1_col[3:]  # strip 'T1_'
    t2_col = f'T2_{stat_name}'
    diff_col = f'Diff_{stat_name}'
    
    if t2_col in mm.columns and diff_col not in mm.columns:
        mm[diff_col] = mm[t1_col] - mm[t2_col]
        new_diff_cols.append(diff_col)

print(f'Created {len(new_diff_cols)} Diff_ columns from T1/T2 pairs:')
for c in sorted(new_diff_cols):
    nan_count = mm[c].isna().sum()
    corr = mm[c].corr(mm['Target_Win']) if 'Target_Win' in mm.columns else float('nan')
    print(f'  {c:<35s}  corr={corr:+.4f}  NaN={nan_count}')

## Elo Win Probability

In [ ]:
if 'Elo_Win_Prob' not in mm.columns:
    hca = np.where(mm['Is_Home'].astype(float) == 1, ELO_HOME_ADV, -ELO_HOME_ADV)
    mm['Elo_Win_Prob'] = 1.0 / (1.0 + 10 ** (-((mm['Diff_Pregame_Elo'].fillna(0) + hca)) / 400.0))
    print(f"Elo_Win_Prob computed. Range: [{mm['Elo_Win_Prob'].min():.3f}, {mm['Elo_Win_Prob'].max():.3f}]")
    if 'Target_Win' in mm.columns:
        corr = mm['Elo_Win_Prob'].corr(mm['Target_Win'])
        print(f"Elo_Win_Prob corr with Target_Win: {corr:.4f}")
else:
    print('Elo_Win_Prob already present.')

## Save matchups_v2.csv

In [ ]:
# Sort by season / date / game
sort_cols = [c for c in ['Season', 'DayDate', 'GameID'] if c in mm.columns]
mm = mm.sort_values(sort_cols).reset_index(drop=True)

out_path = 'matchups_v2.csv'
mm.to_csv(out_path, index=False)
print(f'Wrote {out_path}  ({len(mm):,} rows, {len(mm.columns)} cols)')

# Summary of key columns
print(f'\n=== Dataset summary ===')
if 'Season' in mm.columns:
    print(f'Seasons: {sorted(mm["Season"].unique())}')
print(f'Rows:    {len(mm):,}')
diff_in_output = [c for c in mm.columns if c.startswith('Diff_')]
print(f'\nDiff_ features ({len(diff_in_output)}):')
for c in sorted(diff_in_output):
    if 'Target_Win' in mm.columns:
        corr = mm[c].corr(mm['Target_Win'])
        print(f'  {c:<35s}  corr={corr:+.4f}')
    else:
        print(f'  {c}')

print(f'\nTop 10 features by |corr| with Target_Win:')
if 'Target_Win' in mm.columns:
    num_cols = [c for c in mm.columns 
                if pd.api.types.is_numeric_dtype(mm[c]) 
                and c not in ['Target_Win','Is_Home','GameID','Season']]
    corrs = mm[num_cols].corrwith(mm['Target_Win']).abs().sort_values(ascending=False)
    print(corrs.head(10).to_string())